# 10. Stability of the exponent γ with the common 2PCF chain

**Question.** Calibration and comparison now use the same 2PCF chain. What is left of
the gap between γ_cal (Table 1, complete dataset) and γ measured on the comparison
extract (N = 3 000)? How much does the minimum-pairs criterion move γ? Is the exponent
of the mixture stable over realisations, and are the ranks of Table 2 stable?

**Data.** The nine datasets, cache of `scripts/compute_all.py` for the ranks.
**Outputs.** `out/10_gamma_consistency.csv`, `out/10_mixture_realisations.csv`.
**Paper.** Sec. 4.2 (exponent of the mixture, ranks over 10 realisations) and
Limitations.

History. Until November 2026 the comparison had its own estimator (normalised
coordinates, 18 bins from 0.5 × mean NN to 0.55 × largest separation, unweighted fit),
and this notebook measured the gap between the two chains: up to 0.3 on γ, coming from
the window and the bins, not from the extract (commit `8d18f31`). The comparison chain
is gone, so this gap has no object any more. What remains to check is listed above.

Running time about one hour, mostly the 45 mixtures of section 3.


## Conventions of the common chain (2PCF, fit window, amplitude ratio, SP)

Since the discussion with Charles (November 2026) every notebook uses one chain,
the one of the calibration (`src/calibration.py`, `src/sp_helpers.py`).

**2PCF.** `methods_two_point_correlation.PCF_with_variance`, called here through
`synthesis_utils.compute_2pcf`. Distances in metres, 20 logarithmic edges from the mean
nearest-neighbour distance of the point set to r_max of the dataset, ξ attached to the
upper edge of the bin, Landy-Szalay with split random catalogues (5 × 7 000 points drawn
inside the border), k = 3 repetitions whose spread gives the error bars.

**Fit window and minimum number of pairs.** The power law ξ(r) = (r/r0)^(−γ) is fitted
on the bins 1 to 18 (bin 0 lies below the mean nearest-neighbour distance, the last one
touches r_max), weighted by the spread over the repetitions. One more rule removes
small-r bins: starting from the smallest r, a bin is dropped as long as it holds fewer
than `MIN_PAIRS = 100` data pairs (DD). The first bin with DD ≥ 100 opens the window and
all the bins above it are kept, so the rule only moves the lower bound. Reason: the
relative Poisson noise of a bin is 1/√DD (10 % at 100 pairs). Below that, ξ of the bin
is set by a handful of pairs and can even be negative, and such bins sit at the small r
where the fit has its lever arm on γ. The window actually used is printed by
`su.describe_fit_window(pcf)`.

**Amplitude ratio.** log a = ⟨log ξ_data − log ξ_SP⟩ over the bins of the same window
where both are positive, then n_rand = m_SP (a^(−1/2) − 1). The mean of logs is
symmetric in data and model and is not driven by one bin where ξ_SP is close to zero,
which the former ρ = ⟨ξ_data/ξ_SP⟩ was. γ and a are measured on the same bins.

**SP generation.** `sp_helpers.soneira_peebles_nd` with `dim = 2`: a child that falls
outside the border is drawn again alone (up to 64 times), so the tree keeps its η^L
leaves, is not truncated, and the generation ends whatever the size of the territory.


In [1]:
import sys
import time
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.spatial import cKDTree

import synthesis_utils as su
import calibration as cal

N_CMP = 3000                     # size of the comparison extract (paper)
SEEDS = [42, 43, 44, 45, 46]     # realisations of the mixture (42 is the one of the paper)
N_RUNS = 5                       # runs of the 2PCF on the same extract (unseeded catalogues)
MP_GRID = [0, 100, 300, 1000]    # thresholds on DD (0 = criterion off, 100 = default)
OUT = ROOT / "out"

names = [n for n, ds in su.DATASETS.items()
         if Path(ds["path_border"]).exists() and Path(ds["path_csv"]).exists()]
print("datasets found:", names)

datasets found: ['Austria Cities', 'Belgium Cities', 'RTE France', 'Gares France', 'ENEDIS France', 'France Communes', 'France Routes', 'Germany Stops', 'Swiss Stops']


## 1. γ on the comparison extract, and the effect of the minimum-pairs criterion

On the extract of notebook 07 (seed 42, N = 3 000 or the complete set if smaller), the
2PCF is computed `N_RUNS` times, since the random catalogues are not seeded: the spread
of γ over the runs is the uncertainty coming from the catalogue alone. γ_cal is the
value of Table 1, measured on the complete dataset.

The criterion is then varied on one and the same 2PCF (first run): `MIN_PAIRS` = 0
(off), 100 (default), 300 and 1 000. Only the lower bound of the window moves. The
table also gives the number of data pairs DD in the first bins of the window, which is
what the criterion looks at.


In [2]:
def _g(fit):
    return np.nan if fit["gamma"] is None else float(fit["gamma"])


rows, curves, loaded = [], {}, {}
for name in names:
    t0 = time.time()
    ds = dict(su.DATASETS[name])
    ld = su.load_dataset(ds, compute_data_2pcf=False, verbose=False)
    N = min(N_CMP, ld["N_real"])
    if N < ld["N_real"]:              # same extract as su.data_2pcf_at_N(ld, N, seed=42)
        idx = np.random.default_rng(su.SEED).choice(ld["N_real"], size=N, replace=False)
        xy_d = ld["xy_norm"][idx]
    else:
        xy_d = ld["xy_norm"]
    runs = [su.compute_2pcf(xy_d, ld, full=True) for _ in range(N_RUNS)]
    g = np.array([_g(p) for p in runs])
    pcf = runs[0]
    w = pcf["fit_info"]
    row = dict(dataset=name, N=N, gamma_cal=ds["gamma_data"],
               gamma_extract=np.nanmean(g), gamma_std=np.nanstd(g, ddof=1),
               gamma_min=np.nanmin(g), gamma_max=np.nanmax(g),
               bins_dropped=len(w["dropped_min_pairs"]),
               r_lo_km=w["r_lo"] / 1e3, r_hi_km=w["r_hi"] / 1e3,
               DD_bin1=pcf["dd"][1], DD_bin2=pcf["dd"][2], DD_bin3=pcf["dd"][3])
    for mp in MP_GRID:                # same 2PCF, only the window changes
        f = su.fit_powerlaw(pcf, min_pairs=mp)
        row[f"gamma_mp{mp}"] = _g(f)
        row[f"dropped_mp{mp}"] = len(f["fit_info"]["dropped_min_pairs"])
    rows.append(row)
    curves[name] = pcf
    loaded[name] = (ds, ld, N, xy_d, pcf)
    print(f"{name:16s} done in {time.time() - t0:4.0f} s   {su.describe_fit_window(pcf)}")

tab = pd.DataFrame(rows).set_index("dataset")
tab["extract - cal"] = tab.gamma_extract - tab.gamma_cal
tab["(extract - cal) / std"] = tab["extract - cal"] / tab.gamma_std
pd.set_option("display.width", 250)
print()
print(tab[["N", "gamma_cal", "gamma_extract", "gamma_std", "gamma_min", "gamma_max",
           "extract - cal", "(extract - cal) / std"]].round(3).to_string())
print("\nminimum-pairs criterion on the first run (γ, and number of small-r bins dropped):")
print(tab[["DD_bin1", "DD_bin2", "DD_bin3", "bins_dropped", "r_lo_km"]
          + [f"gamma_mp{mp}" for mp in MP_GRID]
          + [f"dropped_mp{mp}" for mp in MP_GRID]].round(3).to_string())

Austria Cities   done in  106 s   fit window: bins 1..18 (3.95-235.9 km), 0 small-r bin(s) dropped (DD < 100)


Belgium Cities   done in   80 s   fit window: bins 1..18 (4.42-161.8 km), 0 small-r bin(s) dropped (DD < 100)


RTE France       done in   95 s   fit window: bins 1..18 (6.06-133.4 km), 0 small-r bin(s) dropped (DD < 100)


Gares France     done in  122 s   fit window: bins 1..18 (5.78-133.0 km), 0 small-r bin(s) dropped (DD < 100)


ENEDIS France    done in  113 s   fit window: bins 1..18 (7.14-134.6 km), 0 small-r bin(s) dropped (DD < 100)


France Communes  done in   87 s   fit window: bins 1..18 (7.92-135.4 km), 0 small-r bin(s) dropped (DD < 100)


France Routes    done in   82 s   fit window: bins 1..18 (7.35-134.8 km), 0 small-r bin(s) dropped (DD < 100)


Germany Stops    done in  105 s   fit window: bins 1..18 (5.74-133.0 km), 0 small-r bin(s) dropped (DD < 100)


Swiss Stops      done in  121 s   fit window: bins 1..18 (1.45-123.2 km), 0 small-r bin(s) dropped (DD < 100)

                    N  gamma_cal  gamma_extract  gamma_std  gamma_min  gamma_max  extract - cal  (extract - cal) / std
dataset                                                                                                               
Austria Cities   2253      0.789          0.798      0.067      0.718      0.881          0.009                  0.132
Belgium Cities    815      0.808          0.892      0.119      0.715      1.017          0.084                  0.707
RTE France       3000      0.931          0.987      0.005      0.980      0.993          0.056                 10.635
Gares France     3000      0.643          0.772      0.027      0.727      0.793          0.129                  4.834
ENEDIS France    3000      0.954          0.920      0.045      0.856      0.978         -0.034                 -0.742
France Communes  3000      0.254          0.296      0.0

- With one chain, γ of the extract is within 0.06 of γ_cal on seven datasets. The two
  exceptions are the Belgian cities (+0.08, below one standard deviation of the runs)
  and the railway stations (+0.13, 5σ). The gaps of up to 0.3 of the former version are
  gone: they came from the second estimator.
- The spread over five runs of the same extract is 0.03 to 0.07 on seven datasets, 0.12
  on Belgium (815 nodes) and 0.005 on RTE. It comes from the unseeded random catalogues
  alone, and it is the floor of the uncertainty on γ_cal. On RTE the gap of +0.06 is ten
  times this spread: there the extract of 3 000 among 4 932 nodes does differ from the
  complete set.
- Minimum-pairs criterion. At N = 3 000 (815 for Belgium) the first bin of the window
  already holds 370 to 1 360 data pairs, so `MIN_PAIRS = 100` drops no bin on any of the
  nine datasets, and neither does 300. The criterion is a safeguard for smaller samples
  (the high-population subsets of notebook 01, small extracts). With a threshold of
  1 000, one to three bins leave the window: γ moves by at most 0.05 on eight
  datasets, and by +0.29 on Belgium (three bins dropped out of 18), the set with the
  fewest nodes.

## 2. The 2PCF of the extract and its fit window

Open circles: bins of the fit. Red crosses: small-r bins dropped by the criterion
(DD < 100), if any. Line: fitted power law.


In [3]:
fig, axs = plt.subplots(3, 3, figsize=(16, 13))
for ax, name in zip(axs.ravel(), names):
    p = curves[name]
    r = p["r"] / 1e3
    xi_pos = np.where(p["xi"] > 0, p["xi"], np.nan)
    ax.errorbar(r, xi_pos, yerr=p["xi_std"], fmt="o-", c="C0", ms=4, lw=1,
                label="ξ of the extract (± spread of 3 repetitions)")
    m = p["fit_mask"]
    ax.plot(r[m], xi_pos[m], "o", mfc="none", ms=10, c="C0", label="bins of the fit")
    d = p["fit_info"]["dropped_min_pairs"]
    if d:
        ax.plot(r[d], xi_pos[d], "x", c="C3", ms=9, mew=2,
                label=f"dropped, DD < {su.MIN_PAIRS}")
    if p["gamma"] is not None:
        ax.plot(r[m], (p["r"][m] / p["r0_m"]) ** (-p["gamma"]), "-", c="C1", lw=1.5,
                label=f"fit, γ = {p['gamma']:.2f}")
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_title(f"{name}  (N = {p['N']}, γ_cal = {tab.loc[name, 'gamma_cal']:.2f})",
                 fontsize=10)
    ax.set_xlabel("r (km)")
    ax.legend(fontsize=7, loc="lower left")
for ax in axs.ravel()[len(names):]:
    ax.axis("off")
for ax in axs[:, 0]:
    ax.set_ylabel("ξ(r)")
fig.tight_layout()
fig.savefig(OUT / "10_fit_windows.pdf", bbox_inches="tight")
plt.show()

<Figure size 1600x1300 with 9 Axes>

All the fit windows start at the mean nearest-neighbour distance (bin 1) and end one bin
before r_max. The 2PCFs bend down at large r, so a single exponent depends on where the
window ends; this is the same for the data and for the models now that they share the
window.

## 3. Exponent and amplitude of the mixture over realisations

Mixture of notebook 07 (`su.build_sp_mixture`, λ from γ_cal) with 5 seeds. For each
one: the amplitude ratio a (log a on the fit window of the data), the SP fraction it
gives, γ of the mixture with the same chain, and the two JSDs against the data.


In [4]:
mix_rows = []
for name in names:
    ds, ld, N, xy_d, pcf_d = loaded[name]
    t0 = time.time()
    for s in SEEDS:
        xy, info = su.build_sp_mixture(ds, ld, pcf_d, N, seed=s, return_info=True)
        pm = su.compute_2pcf(xy, ld, full=True)
        mix_rows.append(dict(
            dataset=name, seed=s, a=info["a"], log_a=info["log_a"],
            n_bins_a=info["n_bins"],
            sp_fraction=info["m_sp"] / (info["m_sp"] + info["n_rand"]),
            gamma_mix=_g(pm), bins_dropped=len(pm["fit_info"]["dropped_min_pairs"]),
            jsd_2pcf=su.js_divergence(pcf_d["r"], pcf_d["xi"], pm["r"], pm["xi"]),
            jsd_nn=su.nn_comparison(xy_d, {"SP": xy})[1]["SP"]))
    print(f"{name:16s} {len(SEEDS)} mixtures in {time.time() - t0:4.0f} s")
mix = pd.DataFrame(mix_rows)
agg = mix.groupby("dataset", sort=False).agg(
    a_mean=("a", "mean"), a_min=("a", "min"), a_max=("a", "max"),
    sp_fraction=("sp_fraction", "mean"),
    gamma_mix_mean=("gamma_mix", "mean"), gamma_mix_std=("gamma_mix", "std"),
    jsd_2pcf_mean=("jsd_2pcf", "mean"), jsd_nn_mean=("jsd_nn", "mean"))
summary = tab[["N", "gamma_cal", "gamma_extract", "gamma_std", "bins_dropped"]
              + [f"gamma_mp{mp}" for mp in MP_GRID]].join(agg)
summary["mix - extract"] = summary.gamma_mix_mean - summary.gamma_extract
pd.set_option("display.width", 250)
print()
print(summary[["gamma_cal", "gamma_extract", "gamma_mix_mean", "gamma_mix_std",
               "mix - extract", "a_mean", "a_min", "a_max", "sp_fraction",
               "jsd_2pcf_mean", "jsd_nn_mean"]].round(4).to_string())

Austria Cities   5 mixtures in  215 s


Belgium Cities   5 mixtures in  125 s


RTE France       5 mixtures in  105 s


Gares France     5 mixtures in  115 s


ENEDIS France    5 mixtures in  231 s


France Communes  5 mixtures in  226 s


France Routes    5 mixtures in  154 s


Germany Stops    5 mixtures in  212 s


Swiss Stops      5 mixtures in  115 s

                 gamma_cal  gamma_extract  gamma_mix_mean  gamma_mix_std  mix - extract  a_mean   a_min   a_max  sp_fraction  jsd_2pcf_mean  jsd_nn_mean
dataset                                                                                                                                                 
Austria Cities       0.789         0.7979          1.0173         0.0757         0.2194  0.0738  0.0548  0.0980       0.2700         0.0006       0.0479
Belgium Cities       0.808         0.8923          1.1302         0.0911         0.2379  0.1161  0.0641  0.1704       0.3369         0.0013       0.0812
RTE France           0.931         0.9867          0.9135         0.1260        -0.0732  0.1076  0.0869  0.1330       0.3273         0.0007       0.0348
Gares France         0.643         0.7722          0.7520         0.0811        -0.0202  0.4635  0.3877  0.5734       0.6790         0.0015       0.0645
ENEDIS France        0.954         0.9204  

- The mixture is within 0.08 of γ of the extract on RTE, the railway stations, ENEDIS and
  the German stops. It is steeper on the roads (+0.14) and on the Austrian (+0.22) and
  Belgian (+0.24) cities, the Swiss stops (+0.27) and the municipalities (+0.30). The
  spread over seeds is 0.03 to 0.17, so the largest gaps are 2 to 3σ of one realisation.
- The amplitude ratio a stays within a factor 1.3 to 2.7 over the five seeds of a dataset
  (largest on Belgium, 0.064 to 0.170), and it is never outside ]0, 1[. The SP fraction
  it gives goes from 0.27 (Austria) to 0.68 (railway stations).
- The 2PCF JSD of the mixture is 0.0002 to 0.0015. These values cannot be compared with
  those of the former chain (other bins).

## 4. Ranks over 10 realisations

Table 2 uses one realisation of each model (seed 42). The cache also holds the mean JSD
over 10 independent realisations (`band_*`). For each dataset the models are ranked on
this mean, and the ranks are averaged over the datasets.

The cache read here was written by `scripts/compute_all.py` with the former comparison
chain. It has to be rebuilt with the common chain (BSTPP needed) before these ranks can
be quoted.


In [5]:
MODELS = ["SP", "Poisson", "LGCP", "Hawkes", "Cox Hawkes"]
single, band, nn = {}, {}, {}
for name in names:
    f = su.CACHE_DIR / f"{su.cache_slug(name)}.npz"
    if not f.exists():
        continue
    d = np.load(f, allow_pickle=True)
    single[name] = {m: float(d[f"{m}_jsd"]) for m in MODELS}
    band[name] = {m: float(d[f"band_{m}_jsd_mean"]) for m in MODELS}
    nn[name] = {m: float(d[f"nn_jsd_{m}"]) for m in MODELS}
if single:
    ranks = pd.DataFrame({
        "2PCF, one realisation (Table 2)": pd.DataFrame(single).T.rank(axis=1).mean(),
        "2PCF, mean of 10 realisations": pd.DataFrame(band).T.rank(axis=1).mean(),
        "NN distances, one realisation": pd.DataFrame(nn).T.rank(axis=1).mean(),
    })
    print(f"mean JSD rank over {len(single)} datasets (1 = best):")
    print(ranks.round(2).to_string())
    first = {k: int((pd.DataFrame(v).T.rank(axis=1)["SP"] == 1).sum())
             for k, v in [("one realisation", single), ("10 realisations", band)]}
    print("\nSP ranked first on the 2PCF:", first)
else:
    print("no cache found, run scripts/compute_all.py first")

mean JSD rank over 9 datasets (1 = best):
            2PCF, one realisation (Table 2)  2PCF, mean of 10 realisations  NN distances, one realisation
SP                                     1.78                           2.22                           2.33
Poisson                                4.56                           3.78                           3.89
LGCP                                   2.78                           2.44                           2.67
Hawkes                                 2.67                           4.11                           4.00
Cox Hawkes                             3.22                           2.44                           2.11

SP ranked first on the 2PCF: {'one realisation': 5, '10 realisations': 4}


In [6]:
out = summary.copy()
out.to_csv(OUT / "10_gamma_consistency.csv")
mix.to_csv(OUT / "10_mixture_realisations.csv", index=False)
print("written: out/10_gamma_consistency.csv, out/10_mixture_realisations.csv")

written: out/10_gamma_consistency.csv, out/10_mixture_realisations.csv


## Results

1. With the common chain, γ of the comparison extract agrees with γ_cal of Table 1 within
   0.06 on seven of the nine datasets (+0.08 on Belgium, +0.13 on the railway stations).
   The run-to-run spread from the random catalogues is 0.03 to 0.12.
2. `MIN_PAIRS = 100` drops no bin on the nine datasets at N = 3 000. A threshold of 1 000
   moves γ by at most 0.05, except on Belgium (+0.29).
3. The mixture reproduces γ of the extract within 0.08 on four datasets and is steeper by
   0.14 to 0.30 on the five others. The amplitude ratio in log varies by a factor 1.3 to
   2.7 over the seeds and never needs clipping.
4. Ranks (former cache, to be rebuilt with the common chain): SP has the best mean rank on
   the 2PCF with one realisation (1.78) and with ten (2.22), first on 5 and 4 datasets.